GroupAttention

In [2]:
import math
import torch
import torch.nn as nn

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self,hidden_dim,num_heads,nums_key_value_heads):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.nums_key_value_heads = nums_key_value_heads
        self.head_dim = hidden_dim // num_heads

        # 初始化q，k，v，o
        self.q = nn.Linear(hidden_dim,num_heads * self.head_dim)
        self.k = nn.Linear(hidden_dim,nums_key_value_heads*self.head_dim)
        self.v = nn.Linear(hidden_dim,nums_key_value_heads*self.head_dim)
        self.o = nn.Linear(hidden_dim,hidden_dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self,x,attention_mask = None):
        # 1.获取batch和seq_len
        batch_size,seq_len,_ = x.size()

        # 2.q，k，v转换
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        # 此时的k、v都是持久化存储在显存中，作为KV Cache的一部分

        # q - [batch_size,seq_len,num_heads*self.head_dim]
        q = q.view(batch_size,seq_len,self.num_heads,self.head_dim)
        k = k.view(batch_size, seq_len, self.nums_key_value_heads, self.head_dim)
        v = v.view(batch_size, seq_len, self.nums_key_value_heads, self.head_dim)

        # 关注: nums_head 和 nums_key_value_head 的关系
        q = q.transpose(1, 2) # (b, nums_head, seq, head_dim)
        k = k.transpose(1, 2) # (b, nums_key_value_head, seq, head_dim)
        v = v.transpose(1, 2)  # (b, nums_key_value_head, seq, head_dim)

         # k v repeat； （广播操作）
         # repeat操作：复制临时对象，计算只在这一步发生，计算完成后就会被丢弃

        k = k.repeat_interleave(self.num_heads // self.nums_key_value_heads, dim=1)
        v = v.repeat_interleave(self.num_heads // self.nums_key_value_heads, dim=1)
        print(k.shape)
        print(q.shape)
        # 3.矩阵乘法
        attention_weight = (q @ k.transpose(-1,-2)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0,
                float("-1e20")
            )

        # 4.softmax
        attention_weight = torch.softmax(attention_weight,dim=-1)
        attention_weight = self.dropout(attention_weight)

        # 5.output
        output = attention_weight @ v
        output = output.transpose(1,2).contiguous()
        output = output.view(batch_size,seq_len,-1)
        
        return self.o(output)

In [18]:
x = torch.rand(3, 2, 128)
net = GroupQueryAttention(128, 8, 4)
net(x).shape

torch.Size([3, 8, 2, 16])
torch.Size([3, 8, 2, 16])


torch.Size([3, 2, 128])